# GBDF Gender Labels — Download + Commit

Small standalone notebook: grabs the single `GBDF_training_labels.xlsx` (gender annotations,
the accuracy-gap fairness baseline) from the GBDF GitHub release, verifies it, and commits.

GBDF is gender-only — your **primary** fairness labels are the Xu 65.3M annotations (already downloaded:
A-FF++, A-Celeb-DF with age/gender/ethnicity). GBDF is just the baseline-for-contrast your plan calls for.


## Cell 1 — Mount + paths + git creds

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, subprocess
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
PARENT = "/content/drive/MyDrive/CDTS_Research"
ANNO = f"{REPO}/data/annotations/GBDF"
os.makedirs(ANNO, exist_ok=True)
# restore git creds from Drive
for f in [".git-credentials", ".gitconfig"]:
    src = f"{PARENT}/{f}"
    if os.path.exists(src): subprocess.run(f'cp "{src}" /root/{f}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)
print("REPO exists:", os.path.isdir(REPO))
print("GBDF folder:", ANNO)

Mounted at /content/drive
REPO exists: True
GBDF folder: /content/drive/MyDrive/CDTS_Research/deepfake-trust-research/data/annotations/GBDF


## Cell 2 — Download the GBDF xlsx

Tries the standard release-asset filename. If it fails, the asset is named slightly differently —
check https://github.com/aakash4305/GBDF/releases/tag/v1.0 under "Assets" for the exact name.


In [2]:
import os, subprocess, glob
# candidate filenames (README says GBDF_training_labels.xlsx; try variants)
candidates = [
    "GBDF_training_labels.xlsx",
    "GBDF_testing_labels.xlsx",
    "GBDF_labels.xlsx",
]
base = "https://github.com/aakash4305/GBDF/releases/download/v1.0"
got = []
for fname in candidates:
    out = f"{ANNO}/{fname}"
    subprocess.run(f'wget -q -O "{out}" "{base}/{fname}"', shell=True)
    sz = os.path.getsize(out) if os.path.exists(out) else 0
    if sz > 1000:
        print(f"OK  {fname}: {sz/1024:.0f} KB")
        got.append(fname)
    else:
        print(f"--  {fname}: not found ({sz} bytes)")
        if os.path.exists(out): os.remove(out)
print("\nGBDF xlsx files:", [os.path.basename(f) for f in glob.glob(f"{ANNO}/*.xlsx")])
if not got:
    print("\n>>> None downloaded. Open the release page, check Assets for the exact xlsx name,")
    print(">>> then edit the candidates list above with the correct filename.")

OK  GBDF_training_labels.xlsx: 174 KB
--  GBDF_testing_labels.xlsx: not found (0 bytes)
--  GBDF_labels.xlsx: not found (0 bytes)

GBDF xlsx files: ['GBDF_training_labels.xlsx']


## Cell 3 — Inspect the xlsx (confirm it's real gender-label data)

In [3]:
import glob, os
import pandas as pd
xlsx = glob.glob(f"{ANNO}/*.xlsx")
if xlsx:
    df = pd.read_excel(xlsx[0])
    print("file:", os.path.basename(xlsx[0]))
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print("\nfirst rows:")
    print(df.head(5).to_string())
else:
    print("no xlsx to inspect - run Cell 2 first")

file: GBDF_training_labels.xlsx
shape: (6999, 5)
columns: ['Filename', 'Unnamed: 1', 'Dataset', 'Real', 'Gender']

first rows:
       Filename  Unnamed: 1   Dataset  Real Gender
0  id0_0000.mp4         NaN  Celeb-DF     1      M
1  id0_0002.mp4         NaN  Celeb-DF     1      M
2  id0_0003.mp4         NaN  Celeb-DF     1      M
3  id0_0004.mp4         NaN  Celeb-DF     1      M
4  id0_0005.mp4         NaN  Celeb-DF     1      M


## Cell 4 — Gitignore + commit

The xlsx is small (likely <1MB) so it CAN go in git. But annotations are already gitignored
(`data/annotations/`), so we force-add this one file with `-f`, OR just leave GBDF Drive-only
like the Xu annotations. Default here: commit it (it's tiny and useful to version).


In [4]:
import os, glob, subprocess
os.chdir(REPO)
xlsx = glob.glob(f"{ANNO}/*.xlsx")
if xlsx:
    # force-add past the data/annotations/ gitignore (file is tiny)
    for x in xlsx:
        subprocess.run(f'git add -f "{x}"', shell=True)
    subprocess.run('git status --short', shell=True)
    print("\n(review staged above; run Cell 5 to commit)")
else:
    print("no xlsx - nothing to commit")


(review staged above; run Cell 5 to commit)


## Cell 5 — Commit + push

In [5]:
import os, subprocess
os.chdir(REPO)
subprocess.run('git commit -m "Add GBDF gender labels (accuracy-gap fairness baseline)"', shell=True)
subprocess.run('git push origin main 2>&1 | tail -4', shell=True)
print("\ncommit hash:")
subprocess.run('git rev-parse HEAD', shell=True)


commit hash:


CompletedProcess(args='git rev-parse HEAD', returncode=0)

In [ ]:
import os, subprocess
os.chdir(REPO)
r = subprocess.run("git log --oneline -2", shell=True, capture_output=True, text=True)
print("=== recent commits ===")
print(r.stdout)
r2 = subprocess.run("git status --short", shell=True, capture_output=True, text=True)
print("=== uncommitted ===")
print(r2.stdout if r2.stdout.strip() else "(clean - all committed)")
r3 = subprocess.run("git log --oneline origin/main -1", shell=True, capture_output=True, text=True)
print("=== latest on GitHub ===")
print(r3.stdout)

=== recent commits ===
ae1d6bf Add Celeb-DF-v2 test split: 518 videos (16420 frames, 124 identities) + leakage-safe manifest + JSON
8eb1803 Close NB01: complete FF++ c23 dataset (159627 frames, 1000 identities) + leakage-safe manifest; fix build_manifest_from_json method extraction; capture DeepfakeBench patches

